# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [17]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
 
MODEL = "gemini-2.5-flash-lite"
 
groq_api_key = os.getenv('GOOGLE_API_KEY')
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

groq = OpenAI(api_key=groq_api_key, base_url=gemini_url)

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [00:54<00:00, 18.30s/it]


In [4]:
len(deals)

30

In [5]:
deals[10].describe()

'Title: 6th-Gen. Apple iPad 9.7" 32GB WiFi Tablet for $77 + free shipping\nDetails: Use promo code "VIPOUTLETMARCH" to drop the price to the lowest we\'ve ever seen. The coupon expires on April 1 at 3am ET. Buy Now at eBay\nFeatures: \nURL: https://www.dealnews.com/6-th-Gen-Apple-iPad-9-7-32-GB-Wi-Fi-Tablet-for-77-free-shipping/21813948.html?iref=rss-c39'

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [6]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [7]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [8]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Samsung Q-series 11.1.4-Channel Soundbar w/ Subwoofer & Rear Speaker for $898 + free shipping w/ Pri
Details: That's $602 less than our mention from last August and the best deal we've seen for this model. This deal expires on March 29. Buy Now at Woot! An Amazon Company
Features: 
URL: https://www.dealnews.com/Samsung-Q-series-11-1-4-Channel-Soundbar-w-Subwoofer-Rear-Speaker-f

In [18]:
response = groq.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="high")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description="The Mova MOVA Z60 Ultra Roller Complete Robot Vacuum is engineered with a powerful 28,000Pa suction capability for exceptional cleaning performance. It features an innovative AutoShield carpet protection system that deploys a shield to prevent moisture damage on carpets during operation. The vacuum also boasts an 'Always Clean' roller system with real-time washing for consistently streak-free floors, and it can climb thresholds up to 8cm, ensuring comprehensive coverage.", price=999.0, url='https://www.dealnews.com/products/Mova/MOVA-Z60-Ultra-Roller-Complete-Robot-Vacuum/495937.html?iref=rss-f1912'), Deal(product_description="This AMD Ryzen 7 5800XT processor is built on the Vermeer architecture utilizing the Zen 3 core, featuring 8 cores with a base clock speed of 3.8GHz for robust desktop performance. It's designed for demanding computing tasks and gaming. As an added bonus, this deal includes a complimentary Rosewill Cordless Air Duste

In [19]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


The Mova MOVA Z60 Ultra Roller Complete Robot Vacuum is engineered with a powerful 28,000Pa suction capability for exceptional cleaning performance. It features an innovative AutoShield carpet protection system that deploys a shield to prevent moisture damage on carpets during operation. The vacuum also boasts an 'Always Clean' roller system with real-time washing for consistently streak-free floors, and it can climb thresholds up to 8cm, ensuring comprehensive coverage.
999.0
https://www.dealnews.com/products/Mova/MOVA-Z60-Ultra-Roller-Complete-Robot-Vacuum/495937.html?iref=rss-f1912

This AMD Ryzen 7 5800XT processor is built on the Vermeer architecture utilizing the Zen 3 core, featuring 8 cores with a base clock speed of 3.8GHz for robust desktop performance. It's designed for demanding computing tasks and gaming. As an added bonus, this deal includes a complimentary Rosewill Cordless Air Duster, enhancing its value.
212.0
https://www.dealnews.com/AMD-Ryzen-7-5800-XT-Vermeer-Zen-3-

In [20]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [21]:
from agents.scanner_agent import ScannerAgent

In [22]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [23]:
result

NameError: name 'result' is not defined

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [27]:
load_dotenv(override=True)

True

In [28]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [29]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with z


In [30]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [31]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [32]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


In [33]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

INFO:root:[Messaging Agent] Messaging Agent is using Claude to craft the message
13:35:00 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= claude-sonnet-4-5; provider = anthropic
INFO:LiteLLM:
LiteLLM completion() model= claude-sonnet-4-5; provider = anthropic



Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



BadRequestError: litellm.BadRequestError: AnthropicException - {"type":"error","error":{"type":"invalid_request_error","message":"Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits."},"request_id":"req_011CYi3nXhgenfH6q3zcLPKh"}